In [23]:
pip install peptides

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 122 kB 2.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from Bio import SeqIO
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ==== Step 1: Amino acid vocab + encoding ====
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_idx = {aa: i+1 for i, aa in enumerate(amino_acids)}  # 0 is reserved for padding
vocab_size = len(aa_to_idx) + 1  # includes padding

def encode_sequence(seq, maxlen):
    idx_seq = [aa_to_idx.get(aa, 0) for aa in seq]
    return idx_seq + [0] * (maxlen - len(idx_seq))

# ==== Step 2: Load model1.csv for regression ====
df_model1 = pd.read_csv("/Users/jiayingyou/Desktop/model1.csv")
seqs_reg = df_model1["Sequence"].tolist()
labels_reg = df_model1["IC50 Biofilm"].values

# ==== Step 3: Load FASTA for classification ====
positive = list(SeqIO.parse("/Users/jiayingyou/Documents/positive.fasta", "fasta"))
negative = list(SeqIO.parse("/Users/jiayingyou/Documents/negative.fasta", "fasta"))
seqs_cls = [str(r.seq) for r in positive + negative]
labels_cls = [1]*len(positive) + [0]*len(negative)

# ==== Step 4: Pad + encode ====
maxlen = max(max(len(s) for s in seqs_reg), max(len(s) for s in seqs_cls))
X1 = torch.tensor([encode_sequence(s, maxlen) for s in seqs_reg], dtype=torch.long)
y1 = torch.tensor(labels_reg, dtype=torch.float32).view(-1, 1)

X2 = torch.tensor([encode_sequence(s, maxlen) for s in seqs_cls], dtype=torch.long)
y2 = torch.tensor(labels_cls, dtype=torch.long)

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

# ==== Step 5: BiLSTM Model ====
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=8, output_dim=1, task='regression'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim*2, 8),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(8, output_dim)
        )
        self.task = task

    def forward(self, x):
        x = self.embedding(x)                          # (batch, seq_len, embed)
        _, (h_n, _) = self.lstm(x)                     # h_n: (2, batch, hidden)
        h = torch.cat((h_n[0], h_n[1]), dim=1)         # (batch, hidden*2)
        out = self.head(h)
        return out

# ==== Step 6: Train Model 1 (Regression) ====
model1 = BiLSTMModel(vocab_size, task='regression')
loss_fn1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)

for epoch in range(300):
    model1.train()
    optimizer1.zero_grad()
    pred = model1(X1_train)
    loss = loss_fn1(pred, y1_train)
    loss.backward()
    optimizer1.step()
    if epoch % 10 == 0:
        print(f"[Reg] Epoch {epoch}, Loss: {loss.item():.4f}")

# ==== Step 7: Save Model 1 weights ====
torch.save(model1.state_dict(), "bilstm_model1_regression.pt")

# ==== Step 8: Train Model 2 (Classification) with shared BiLSTM ====
model2 = BiLSTMModel(vocab_size, output_dim=2, task='classification')
model2.embedding = model1.embedding
model2.lstm = model1.lstm
for param in model2.embedding.parameters(): param.requires_grad = False
for param in model2.lstm.parameters(): param.requires_grad = False

loss_fn2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(filter(lambda p: p.requires_grad, model2.parameters()), lr=0.001)

for epoch in range(200):
    model2.train()
    optimizer2.zero_grad()
    logits = model2(X2_train)
    loss = loss_fn2(logits, y2_train)
    loss.backward()
    optimizer2.step()
    if epoch % 10 == 0:
        print(f"[CLS] Epoch {epoch}, Loss: {loss.item():.4f}")

# ==== Step 9: Evaluate Classification ====
model2.eval()
with torch.no_grad():
    pred_cls = model2(X2_test).argmax(dim=1)
    acc = (pred_cls == y2_test).float().mean().item()
print(f"\n✅ Final Classification Accuracy: {acc:.2f}")
torch.save(model2.state_dict(), "bilstm_model2_classifier.pt")


[Reg] Epoch 0, Loss: 176.5646
[Reg] Epoch 10, Loss: 173.9203
[Reg] Epoch 20, Loss: 171.5634
[Reg] Epoch 30, Loss: 166.6795
[Reg] Epoch 40, Loss: 163.5430
[Reg] Epoch 50, Loss: 156.8339
[Reg] Epoch 60, Loss: 150.2554
[Reg] Epoch 70, Loss: 144.9744
[Reg] Epoch 80, Loss: 135.4356
[Reg] Epoch 90, Loss: 130.3074
[Reg] Epoch 100, Loss: 114.3121
[Reg] Epoch 110, Loss: 101.7571
[Reg] Epoch 120, Loss: 102.0316
[Reg] Epoch 130, Loss: 98.7250
[Reg] Epoch 140, Loss: 99.0990
[Reg] Epoch 150, Loss: 84.1931
[Reg] Epoch 160, Loss: 91.9972
[Reg] Epoch 170, Loss: 84.5493
[Reg] Epoch 180, Loss: 82.9352
[Reg] Epoch 190, Loss: 91.1248
[Reg] Epoch 200, Loss: 83.6802
[Reg] Epoch 210, Loss: 86.0706
[Reg] Epoch 220, Loss: 81.7421
[Reg] Epoch 230, Loss: 82.8870
[Reg] Epoch 240, Loss: 93.4985
[Reg] Epoch 250, Loss: 83.7396
[Reg] Epoch 260, Loss: 83.6301
[Reg] Epoch 270, Loss: 81.7264
[Reg] Epoch 280, Loss: 75.4129
[Reg] Epoch 290, Loss: 75.4873
[CLS] Epoch 0, Loss: 0.6916
[CLS] Epoch 10, Loss: 0.6836
[CLS] Epoch

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from Bio import SeqIO
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ==== Step 1: Amino acid vocab + encoding ====
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_idx = {aa: i+1 for i, aa in enumerate(amino_acids)}  # 0 for padding
vocab_size = len(aa_to_idx) + 1

def encode_sequence(seq, maxlen):
    idx_seq = [aa_to_idx.get(aa, 0) for aa in seq]
    return idx_seq + [0] * (maxlen - len(idx_seq))

# ==== Step 2: Load model1.csv for regression ====
df_model1 = pd.read_csv("/Users/jiayingyou/Desktop/model1.csv")
seqs_reg = df_model1["Sequence"].tolist()
labels_reg = df_model1["IC50 Biofilm"].values

# ==== Step 3: Load FASTA for classification ====

# ==== Step 3: Load FASTA for classification (length = 12 only) ====
positive = [r for r in SeqIO.parse("/Users/jiayingyou/Documents/positive.fasta", "fasta")]
negative = [r for r in SeqIO.parse("/Users/jiayingyou/Documents/negative.fasta", "fasta")]
seqs_cls = [str(r.seq) for r in positive + negative]
labels_cls = [1] * len(positive) + [0] * len(negative)

# ==== Step 4: Pad + encode ====
maxlen = 50  # fixed length
X2 = torch.tensor([encode_sequence(s, maxlen) for s in seqs_cls], dtype=torch.long)
y2 = torch.tensor(labels_cls, dtype=torch.long)

X1 = torch.tensor([encode_sequence(s, maxlen) for s in seqs_reg], dtype=torch.long)
y1 = torch.tensor(labels_reg, dtype=torch.float32).view(-1, 1)



X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, stratify=y2, random_state=42)

# ==== Step 5: BiLSTM Model ====
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=8, output_dim=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim*2, 8),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(8, output_dim)
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat((h_n[0], h_n[1]), dim=1)
        return self.head(h)

# ==== Step 6: Train Model 1 (Regression) ====
model1 = BiLSTMModel(vocab_size)
loss_fn1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)

for epoch in range(300):
    model1.train()
    optimizer1.zero_grad()
    pred = model1(X1_train)
    loss = loss_fn1(pred, y1_train)
    loss.backward()
    optimizer1.step()
    if epoch % 10 == 0:
        print(f"[Reg] Epoch {epoch}, Loss: {loss.item():.4f}")

torch.save(model1.state_dict(), "bilstm_model1_regression.pt")

# ==== Step 7: Train Model 2 (Classification, early stopping) ====
model2 = BiLSTMModel(vocab_size, output_dim=2)
model2.embedding = model1.embedding
model2.lstm = model1.lstm
# Unfreeze for fine-tuning
for param in model2.embedding.parameters(): param.requires_grad = True
for param in model2.lstm.parameters(): param.requires_grad = True

loss_fn2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

best_acc = 0.0
patience = 10
patience_counter = 0

for epoch in range(200):
    model2.train()
    optimizer2.zero_grad()
    logits = model2(X2_train)
    loss = loss_fn2(logits, y2_train)
    loss.backward()
    optimizer2.step()

    model2.eval()
    with torch.no_grad():
        val_preds = model2(X2_test).argmax(dim=1)
        val_acc = (val_preds == y2_test).float().mean().item()

    print(f"[CLS] Epoch {epoch}, Loss: {loss.item():.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model2.state_dict(), "bilstm_model2_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}. Best val acc: {best_acc:.4f}")
            break

# ==== Step 8: Evaluate Best Model ====
model2.load_state_dict(torch.load("bilstm_model2_best.pt"))
model2.eval()
with torch.no_grad():
    final_preds = model2(X2_test).argmax(dim=1)
    acc = (final_preds == y2_test).float().mean().item()
    print(f"\n✅ Final Classification Accuracy: {acc:.2f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y2_test.numpy(), final_preds.numpy()))
    print("\nClassification Report:")
    print(classification_report(y2_test.numpy(), final_preds.numpy()))

# Optional: Flip prediction test
flipped_preds = 1 - final_preds
flipped_acc = (flipped_preds == y2_test).float().mean().item()
print(f"\nFlipped Accuracy: {flipped_acc:.2f}")


ValueError: expected sequence of length 50 at dim 1 (got 55)

In [11]:
X2.shape

torch.Size([340, 12])

In [14]:
import torch
import torch.nn as nn
import pandas as pd

# ==== Define model architecture (must match training) ====
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=8, output_dim=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 8),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(8, output_dim)
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat((h_n[0], h_n[1]), dim=1)
        return self.head(h)

# ==== Amino acid encoding ====
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_idx = {aa: i + 1 for i, aa in enumerate(amino_acids)}  # 0 for padding
vocab_size = len(aa_to_idx) + 1

def encode_sequence(seq, maxlen=12):
    idx_seq = [aa_to_idx.get(aa, 0) for aa in seq]
    return idx_seq + [0] * (maxlen - len(idx_seq))

# ==== Load model ====
model = BiLSTMModel(vocab_size)
model.load_state_dict(torch.load("bilstm_model1_regression.pt"))
model.eval()

# ==== Predict on new sequences ====
# Example: list of new sequences (must be length 12 or shorter)
new_sequences = ["MLIRVRKLWRIL", "GLFDIVKKVVAK"]  # you can replace this with your actual test set
maxlen = 12
encoded = torch.tensor([encode_sequence(s, maxlen) for s in new_sequences], dtype=torch.long)

# Make predictions
with torch.no_grad():
    predictions = model(encoded).view(-1).numpy()

# ==== Print or Save Results ====
for seq, pred in zip(new_sequences, predictions):
    print(f"{seq} -> Predicted IC50: {pred:.2f}")


MLIRVRKLWRIL -> Predicted IC50: 4.35
GLFDIVKKVVAK -> Predicted IC50: 4.20
